In [ ]:
import os
import shutil
from pathlib import Path
from PIL import Image
import cv2
import numpy as np
from tqdm import tqdm

# ==================== CẤU HÌNH ====================
SOURCE_DIR = r"c:\Code\train"
OUTPUT_DIR = r"c:\Code\dataset"
IMAGE_FORMAT = "png"  # Định dạng tốt nhất: png (lossless)

# Tạo cấu trúc thư mục
os.makedirs(f"{OUTPUT_DIR}/images/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/images/val", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/train", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/labels/val", exist_ok=True)

print("📁 Cấu trúc thư mục được tạo!")
print(f"Output: {OUTPUT_DIR}")

# ==================== CHUYỂN ĐỔI ẢNH ====================
image_files = [f for f in os.listdir(SOURCE_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
total = len(image_files)
print(f"\n📊 Tổng số ảnh: {total}")

# Chia train/val (80/20)
train_count = int(total * 0.8)
train_files = image_files[:train_count]
val_files = image_files[train_count:]

def convert_image(src_path, dst_path, target_format='png'):
    """Convert ảnh sang định dạng tốt nhất cho YOLO"""
    try:
        # Đọc ảnh
        img = cv2.imread(src_path)
        if img is None:
            print(f"❌ Lỗi đọc: {src_path}")
            return False
        
        # Compress PNG hoặc lưu JPG với quality cao nếu cần
        if target_format.lower() == 'png':
            cv2.imwrite(dst_path, img, [cv2.IMWRITE_PNG_COMPRESSION, 9])
        else:
            cv2.imwrite(dst_path, img, [cv2.IMWRITE_JPEG_QUALITY, 95])
        
        return True
    except Exception as e:
        print(f"❌ Lỗi convert {src_path}: {e}")
        return False

# Convert Train set
print(f"\n🔄 Convert TRAIN ({len(train_files)} ảnh)...")
train_success = 0
for filename in tqdm(train_files, desc="Train"):
    src = os.path.join(SOURCE_DIR, filename)
    new_name = os.path.splitext(filename)[0] + f".{IMAGE_FORMAT}"
    dst = os.path.join(OUTPUT_DIR, "images/train", new_name)
    if convert_image(src, dst, IMAGE_FORMAT):
        train_success += 1

# Convert Val set
print(f"\n🔄 Convert VAL ({len(val_files)} ảnh)...")
val_success = 0
for filename in tqdm(val_files, desc="Val"):
    src = os.path.join(SOURCE_DIR, filename)
    new_name = os.path.splitext(filename)[0] + f".{IMAGE_FORMAT}"
    dst = os.path.join(OUTPUT_DIR, "images/val", new_name)
    if convert_image(src, dst, IMAGE_FORMAT):
        val_success += 1

print(f"\n✅ Train: {train_success}/{len(train_files)}")
print(f"✅ Val: {val_success}/{len(val_files)}")

# ==================== TẠO data.yaml ====================
yaml_content = f"""
path: {OUTPUT_DIR.replace(chr(92), '/')}  # dataset root
train: images/train
val: images/val

nc: 1  # Số class (thay đổi theo nhu cầu)
names: ['object']  # Tên class

# Cấu hình YOLO tối ưu
imgsz: 640
batch: 16
epochs: 100
device: 0  # GPU device, đặt -1 nếu dùng CPU
"""

yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(yaml_path, 'w') as f:
    f.write(yaml_content.strip())

print(f"\n✅ File data.yaml tạo tại: {yaml_path}")

# ==================== THỐNG KÊ ====================
print("\n" + "="*50)
print("📊 THỐNG KÊ CUỐI CÙNG")
print("="*50)
print(f"✅ Định dạng: {IMAGE_FORMAT.upper()}")
print(f"✅ Train: {train_success} ảnh")
print(f"✅ Val: {val_success} ảnh")
print(f"✅ Tổng: {train_success + val_success}/{total}")
print(f"✅ Output folder: {OUTPUT_DIR}")
print("\n📌 Để train với YOLOv11, chạy:")
print("   from ultralytics import YOLO")
print("   model = YOLO('yolov11n.pt')")
print(f"   results = model.train(data='{yaml_path}', epochs=100, imgsz=640)")


In [1]:
import os
import cv2
from pathlib import Path
from tqdm import tqdm

# ==================== CẤU HÌNH ====================
SOURCE_DIR = r"c:\Code\train"
OUTPUT_DIR = r"c:\Code\train_resized"
MAX_SIZE = 1280  # Roboflow khuyến nghị (có thể giảm xuống 640 nếu muốn file nhỏ hơn)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==================== LẤY DANH SÁCH ẢNH ====================
image_files = [f for f in os.listdir(SOURCE_DIR) 
               if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
print(f"📊 Tổng số ảnh: {len(image_files)}")

# ==================== RESIZE VÀ LƯU ====================
print(f"\n🔄 Resize ảnh (max: {MAX_SIZE}px, giữ tỉ lệ)...\n")

success = 0
for filename in tqdm(image_files, desc="Processing"):
    src_path = os.path.join(SOURCE_DIR, filename)
    
    try:
        # Đọc ảnh
        img = cv2.imread(src_path)
        if img is None:
            continue
        
        h, w = img.shape[:2]
        
        # Tính toán size mới giữ tỉ lệ
        if w > h:
            new_w = MAX_SIZE
            new_h = int(h * MAX_SIZE / w)
        else:
            new_h = MAX_SIZE
            new_w = int(w * MAX_SIZE / h)
        
        # Resize
        img_resized = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LANCZOS4)
        
        # Lưu (giữ format gốc)
        output_path = os.path.join(OUTPUT_DIR, filename)
        cv2.imwrite(output_path, img_resized, [cv2.IMWRITE_JPEG_QUALITY, 95])
        success += 1
        
    except Exception as e:
        print(f"❌ {filename}: {e}")

# ==================== THỐNG KÊ ====================
print(f"\n✅ Xong! {success}/{len(image_files)} ảnh")
print(f"📁 Output: {OUTPUT_DIR}")
print(f"📌 Ready to upload to Roboflow!")


📊 Tổng số ảnh: 435

🔄 Resize ảnh (max: 1280px, giữ tỉ lệ)...



Processing: 100%|██████████| 435/435 [00:15<00:00, 28.62it/s]


✅ Xong! 435/435 ảnh
📁 Output: c:\Code\train_resized
📌 Ready to upload to Roboflow!


In [1]:
import paho.mqtt.client as mqtt
import json

# MQTT Configuration
broker = "45.117.170.179"
port = 1883  # Default MQTT port
topic = "MGSP-V1/F055CF453AB4/cmd"
#topic = "MGLB-V1/2604-01-003/cmd"
#topic = "MGSP-V1/F055CF453AB4/cmd"
message = {
    "request_id": "abe",
    "audio_stream": {
        "url": "http://192.168.10.4:8090/2.mp3"
    }
}


# Create MQTT client
client = mqtt.Client()

# Connect to broker
client.connect(broker, port, 60)

# Publish message
client.publish(topic, json.dumps(message))

# Disconnect
client.disconnect()
print(f"✅ Message published to topic:{client} ")

print("Message sent successfully!")

✅ Message published to topic:<paho.mqtt.client.Client object at 0x000001C5CA2757F0> 
Message sent successfully!


C:\Users\sonng\AppData\Local\Temp\ipykernel_27348\1439474900.py:19: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


In [ ]:
import paho.mqtt.client as mqtt
import json

# MQTT Configuration
broker = "45.117.170.179"
port = 1883  # Default MQTT port
topic = "MGLB-V1/2604-01-003/cmd"
#topic = "MGLB-V1/2604-01-003/cmd"
#topic = "MGSP-V1/F055CF453AB4/cmd"
message = {
    "request_id": "req_203",
    "audio_stream": {
        "control": "normal"
    }
}

# Create MQTT client
client = mqtt.Client()

# Connect to broker
client.connect(broker, port, 60)

# Publish message
client.publish(topic, json.dumps(message))

# Disconnect
client.disconnect()
print(f"✅ Message published to topic:{client} ")

print("Message sent successfully!")

✅ Message published to topic:<paho.mqtt.client.Client object at 0x0000015A68AC74D0> 
Message sent successfully!


C:\Users\sonng\AppData\Local\Temp\ipykernel_4684\132599162.py:20: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()
